# Laboratorio 1

## Ejercicio 1

### Razonamiento Previo: Fundamentos del Hashing Criptográfico

Un hash es una función matemática unidireccional que toma datos de cualquier tamaño y devuelve una cadena de longitud fija. Su regla criptográfica principal es el determinismo: la misma entrada siempre generará exactamente la misma salida.   

SHA-256 (el algoritmo usado en este ejercicio) fue diseñado específicamente para verificar la integridad de datos y archivos de forma extremadamente rápida. Al implementarlo directamente para almacenar contraseñas sin añadir elementos criptográficos adicionales, se comete un error de diseño conceptual: la velocidad, que es una gran ventaja técnica para procesar archivos grandes, se convierte en la principal vulnerabilidad de seguridad. Un atacante con hardware moderno (como tarjetas gráficas) puede aprovechar esta alta velocidad computacional para realizar ataques de fuerza bruta, probando miles de millones de combinaciones de contraseñas por segundo hasta dar con la correcta. 

In [1]:
from cryptography.hazmat.primitives import hashes # Importa libreria necesaria para trabajar con hashess

def sha256_hash(password): # Funcion que transforma la contraseña
 digest = hashes.Hash(hashes.SHA256()) # Crea un objeto de hash SHA256
 digest.update(password.encode()) # Actualiza el objeto de hash con la contraseña codificada en bytes
 return digest.finalize().hex() # Devuelve el hash final en formato hexadecimal

def almacenar_password(password): # Simula el almacenamiento de la contraseña en la base de datos
 return sha256_hash(password)

def verificar_password(password, hash_almacenado): # Compara la contraseña ingresada con el hash almacenado
 return sha256_hash(password) == hash_almacenado

# Registro
hash_usuario = almacenar_password("123456")

# Login
if verificar_password("123456", hash_usuario):
 print("Acceso concedido")


else:
 print("Acceso denegado")

Acceso concedido


**Qué algoritmo hash se está utilizando?**

SHA-256 (Secure Hash Algorithm de 256 bits).

**Las contraseñas se están almacenando en texto plano?**

No. Cuando se llama a almacenar_password("123456"), la contraseña original es procesada y lo que realmente se almacena 
en la variable hash_usuario es la representación en hexadecimal de su hash.

**Aunque no estén en texto plano, ¿ves algún problema de seguridad importante?**

Sí, la implementación es vulnerable. Al no incluir un salt aleatorio, el sistema es susceptible a ataques mediante 
"Rainbow Tables" (tablas precomputadas de hashes) o ataques de diccionario. Además, al usar un algoritmo rápido como 
SHA-256 en lugar de una función derivadora de claves diseñada específicamente para ser costosa computacionalmente 
(como bcrypt, Argon2 o PBKDF2), se facilita enormemente la rotura del hash por fuerza bruta.

## Ejercicio 2

### Razonamiento Previo: Determinismo y Tablas Precomputadas

Este ejercicio expone la consecuencia práctica de utilizar una función hash simple sin un factor de aleatoriedad. Al aplicar la función SHA-256 a dos variables que contienen la misma cadena de texto exacta ("password123"), la matemática dictamina que el bloque hexadecimal resultante será idéntico.   

Comprender esta propiedad determinista es clave para identificar vulnerabilidades estructurales en bases de datos. Si un sistema almacena hashes de esta manera, un atacante que vulnere la base de datos no necesita descifrar el hash de cada usuario uno por uno. En su lugar, el atacante utiliza "Rainbow Tables" (tablas gigantes precomputadas que emparejan millones de contraseñas de uso común con sus respectivos hashes SHA-256). Al cruzar esta tabla con los datos robados, el atacante solo tiene que buscar coincidencias exactas. Si encuentra un hash coincidente, no solo descubre la contraseña en milisegundos, sino que compromete simultáneamente a todos los usuarios del sistema que hayan utilizado esa misma clave.

In [6]:
from cryptography.hazmat.primitives import hashes

def hash_password(pw): # define la función para hashear la contraseña
 digest = hashes.Hash(hashes.SHA256()) #! Error, no incorpora el salt aleatorio, lo que hace que el hash sea vulnerable a ataques de diccionario y rainbow tables
 digest.update(pw.encode()) # Actualiza el objeto de hash con la contraseña codificada en bytes
 return digest.finalize().hex()

pw1 = "password123"

pw2 = "password123"

print(hash_password(pw1)) # Imprime el hash de la primera contraseña

print(hash_password(pw2)) # Imprime el hash de la segunda contraseña, que es la misma que la primera, por lo que el hash será igual

ef92b778bafe771e89245b89ecbc08a44a4e166c06659911881f383d4473e94f
ef92b778bafe771e89245b89ecbc08a44a4e166c06659911881f383d4473e94f


**¿Qué observa? ¿Hay algún problema de seguridad?**

La consola imprime dos veces el mismo valor hexadecimal porque la contraseña es identica. por lo que si un atacante roba la base de datos de de hashes, 
podrá agrupar a los usuarios que tengan el mismo hash y deducir que comparten la misma contraseña. Peor aún, permite el uso de "Rainbow Tables" 
(tablas gigantes precomputadas de hashes de contraseñas comunes). Si el atacante busca ese hash en su tabla y encuentra que corresponde a 
"password123", habrá comprometido a todos los usuarios que usen esa clave en un solo paso.

**¿Qué problema hay aquí con SHA256?**

Su extremada eficiencia y velocidad computacional. Fue diseñado para procesar grandes volúmenes de datos muy rápido (por ejemplo, para verificar 
que un archivo descargado de internet no esté corrupto). En seguridad de contraseñas, la velocidad beneficia al atacante: usando hardware moderno,
especialmente tarjetas gráficas (GPUs), un atacante puede generar miles de millones de hashes SHA-256 por segundo para probar combinaciones 
rápidamente. Un buen algoritmo para contraseñas debe ser intencionalmente lento y costoso para la CPU/memoria.


## Ejercicio 3

### Razonamiento Previo (Vulnerabilidad de Hashes Simples)

Cuando un sistema usa un algoritmo rápido (como SHA-256) sin añadir un valor aleatorio previo (salt), el proceso es completamente determinista. Un atacante que obtenga el hash almacenado puede tomar una lista de contraseñas comunes, calcular el hash de cada una mediante un bucle, y comparar los resultados. Debido a la alta velocidad de SHA-256, se pueden evaluar millones de palabras por segundo hasta encontrar la coincidencia exacta.

In [7]:
with open("diccionario.txt", "r", encoding="utf-8") as file:
    common = file.read().splitlines() 

target = hash_password("incontrastablemente")

for pw in common:
    if hash_password(pw) == target:
        print("Found:", pw)

Found: incontrastablemente


**a) ¿Ha encontrado la contraseña?** 

Sí, el script devolvió "Found: incontrastablemente". El programa comparó exitosamente el hash objetivo con el hash generado a partir de las palabras del documento de texto.   

**b) ¿Qué ataque ha realizado?** 

Ha ejecutado un ataque de diccionario. Este ataque consiste en iterar sistemáticamente sobre un archivo o lista predefinida de palabras de uso frecuente, aplicando la función hash a cada elemento y comparando el resultado con el hash robado hasta hallar una igualdad perfecta.|

## Ejercicio 4

### Razonamiento Previo: Funciones Derivadoras de Claves (KDF)

Para neutralizar el ataque anterior, la criptografía defensiva utiliza herramientas como bcrypt o PBKDF2HMAC. Estas introducen dos mecanismos fundamentales en la ecuación:   

- Inclusión de un Salt: Se genera una cadena de bytes completamente aleatoria que se concatena a la contraseña de cada usuario de forma única antes de procesarla. Esto asegura que dos contraseñas idénticas produzcan hashes distintos, invalidando ataques precomputados y de diccionario directo.   

- Factor de Trabajo (Iteraciones): Obligan a la CPU a recalcular matemáticamente el hash sobre sí mismo miles de veces (por ejemplo, 390.000 iteraciones). Mientras que verificar un solo login tarda una fracción de segundo aceptable, esta penalización de tiempo destruye la viabilidad de un ataque masivo de diccionario, ya que probar millones de combinaciones requeriría años de cómputo. 

In [ ]:
import bcrypt

pw_original = b" incontrastablemente " 
hashed = bcrypt.hashpw(pw_original, bcrypt.gensalt())

# 4a: Repetición del ataque de diccionario
print("\nIniciando ataque de diccionario con bcrypt...")
with open("diccionario.txt", "r", encoding="utf-8") as file:
    diccionario = file.read().splitlines()

encontrada = False
for palabra in diccionario:
    # Convertimos la palabra del txt a bytes para que bcrypt pueda procesarla
    palabra_bytes = palabra.encode('utf-8')
    if bcrypt.checkpw(palabra_bytes, hashed):
        print("Contraseña encontrada:", palabra)
        encontrada = True
        break

if not encontrada:
    print("Contraseña no encontrada en el diccionario.")

b'$2b$12$MA5DJobSEOiNPz33pTSHQOUD38wYIVAwzYfX10cQI8zj/qT1/iztK'
True


In [8]:
import os
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

# 1. Generar un salt aleatorio de 16 bytes
salt = os.urandom(16)

# 2. Configurar la función derivadora de claves (KDF)
kdf = PBKDF2HMAC(
    algorithm=hashes.SHA256(),
    length=32,
    salt=salt,
    iterations=390000, # Alto costo computacional para frenar ataques
)

# 3. Derivar la clave (equivalente a hashear)
password_bytes = b"incontrastablemente"
clave_segura = kdf.derive(password_bytes)

print("\nHash seguro con PBKDF2HMAC (hex):", clave_segura.hex())


Hash seguro con PBKDF2HMAC (hex): 8423fb7a7261a0f5b81a7c4c089f03372a023708b0a9448308ad451830592083


**Con la salida del código anterior, repita lo realizado en 3)**

Para intentar un ataque de diccionario aquí, tendrías que leer el archivo de texto, convertir cada palabra a bytes (.encode()) y usar la función 
bcrypt.checkpw(palabra_diccionario, hashed) en un bucle for para ver si alguna arroja True.

**b) ¿Es capaz de obtener la contraseña? Justifique su respuesta**

No (o sería extremadamente ineficiente). Hay dos razones. A nivel criptográfico, bcrypt es intencionalmente lento; verificar miles de palabras del 
diccionario tomará una cantidad de tiempo y procesamiento brutal comparado con SHA-256, deteniendo el ataque de fuerza bruta. Además, a nivel de 
código, la variable original incluye espacios (b" incontrastablemente "), por lo que las palabras exactas de tu archivo diccionario.txt nunca 
coincidirán.   
 
**c) ¿Cómo podría obtener el mismo código (a nivel de seguridad) pero con la librería cryptography?**

Se lograría utilizando una función de derivación de claves basada en contraseñas (PBKDF) que incluya un salt aleatorio y un alto número de 
iteraciones. En la librería cryptography, esto se hace usando el módulo PBKDF2HMAC. El proceso requiere usar os.urandom(16) para generar el salt, 
definir un número de iteraciones (por ejemplo, 390000), usar un algoritmo base como SHA-256 internamente, y finalmente usar la función derive() 
para obtener la clave segura.